In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
import itertools as itools
import time
import numpy as np
from collections import defaultdict
from scipy.optimize import linprog
import chaospy as cp
from typing import List, Callable, Union
from numpy.polynomial.legendre import leggauss
import pandas as pd
from pyomo.environ import *
import math
from gurobipy import nlfunc

In [2]:
demand_theta_nominal = {
    'B': 7,
    'C': 4,
}

demand_theta_stddev = {
    'B': (0.3)**(0.5),
    'C': (0.3)**(0.5),
}

t_bounds = [
    (demand_theta_nominal['B'] - 4*demand_theta_stddev['B'], demand_theta_nominal['B'] + 4*demand_theta_stddev['B']),
    (demand_theta_nominal['C'] - 4*demand_theta_stddev['C'], demand_theta_nominal['C'] + 4*demand_theta_stddev['C']),
]

d_bounds = [(15, 25), (14, 26)]
nt = len(t_bounds)
nd = len(d_bounds)
y_dict = {
    (1,1): 0.64,
    (1,0.5): 0.08,
    (1,0): 0.08,
    (0.5,1): 0.12,
    (0.5,0.5): 0.015,
    (0.5,0): 0.015,
    (0,1): 0.04,
    (0,0.5): 0.005,
    (0,0): 0.005,
}

def create_flexibility_model(tbounds:list, dbounds:list, y_list: tuple = None):
    a_rxn = 0.99218
    
    if y_list is None:
        y_list = [1]*len(dbounds)
    m = MPModeler()
    
    u = m.add_var(name='u')
    ma = m.add_var(name='ma')
    
    t_demand_b = m.add_param('t_demand_b')
    t_demand_c = m.add_param('t_demand_c')
    d_pipe = m.add_param('d_pipe')
    d_volume = m.add_param('d_volume')
    
    m.add_constr(-ma + 0.2 * a_rxn * y_list[1] * d_volume <= u)
    m.add_constr(ma - a_rxn * y_list[1] * d_volume <= u)
    m.add_constr(ma - y_list[0] * d_pipe <= u)
    m.add_constr(-30 -ma + 0.8 * a_rxn * y_list[1] * d_volume <= u)
    m.add_constr(-0.6 * ma + t_demand_b <= u)
    m.add_constr(-0.4 * ma + t_demand_c <= u)
    
    m.add_constr(t_bounds[0][0] <= t_demand_b)
    m.add_constr(t_demand_b <= tbounds[0][1])
    m.add_constr(tbounds[1][0] <= t_demand_c)
    m.add_constr(t_demand_c <= tbounds[1][1])

    m.add_constr(dbounds[0][0] <= d_pipe)
    m.add_constr(d_pipe <= dbounds[0][1])
    m.add_constr(dbounds[1][0] <= d_volume)
    m.add_constr(d_volume <= dbounds[1][1])
    
    m.set_objective(u)
    
    return m

def joint_pdf(theta: list, eps:float = 1e-6):
    theta_b, theta_c = theta
    b_nom, c_nom = demand_theta_nominal['B'], demand_theta_nominal['C']
    b_std, c_std = demand_theta_stddev['B'], demand_theta_stddev['C']
    
    contrib_b = (1/np.sqrt(2*np.pi)) * (1/b_std) * np.exp(-(theta_b - b_nom)**2/(2 * b_std**2))
    
    contrib_c = (1/np.sqrt(2*np.pi)) * (1/c_std) * np.exp(-(theta_c - c_nom)**2/(2 * c_std**2))
    
    return contrib_b * contrib_c

def pdf_builder(theta, n, eps=1e-6):
    theta_b = theta[(0, n)]
    theta_c = theta[(1, n)]
    b_nom, c_nom = demand_theta_nominal['B'], demand_theta_nominal['C']
    b_std, c_std = demand_theta_stddev['B'], demand_theta_stddev['C']
    
    contrib_b = (1/np.sqrt(2*math.pi)) * (1/b_std) * nlfunc.exp(-((theta_b - b_nom) * (theta_b - b_nom))/(2 * b_std * b_std))
    
    contrib_c = (1/np.sqrt(2*math.pi)) * (1/c_std) * nlfunc.exp(-((theta_c - c_nom) * (theta_c - c_nom))/(2 * c_std * c_std))
    
    return contrib_b * contrib_c

In [5]:
model = create_flexibility_model(tbounds=t_bounds, dbounds=d_bounds)

In [6]:
prob = model.formulate_problem()
prob.process_constraints()

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-12


In [9]:
solution_flexibility = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.combinatorial)

In [10]:
len(solution_flexibility.critical_regions)

4